In [1]:
from plntxps import *
import pandas as pd
import lmfit
import re
from lmfitxps import models

data processed with plnt_xps version 0.1.1
our crew is replaceable, your package isn't!


In [2]:
def format_satellite_name(name):
    formatted_name = re.sub(r" ", "_", name)
    formatted_name = re.sub(r",", "", formatted_name)
    return formatted_name

satellites = pd.read_csv('example data/perkin elmer mg satellites.csv',
                sep = '\t').to_dict(orient='index')

for satellite in satellites.values():
    satellite['name'] = format_satellite_name(satellite['name'])

In [3]:
satellites

{0: {'name': 'alpha_1_2', 'position': 0.0, 'intensity': 0.0},
 1: {'name': 'alpha_3', 'position': 8.4, 'intensity': 8.0},
 2: {'name': 'alpha_4', 'position': 10.1, 'intensity': 4.1},
 3: {'name': 'alpha_5', 'position': 17.6, 'intensity': 0.6},
 4: {'name': 'alpha_6', 'position': 20.3, 'intensity': 0.5},
 5: {'name': 'beta', 'position': 48.7, 'intensity': 0.5}}

In [4]:
peak = models.ConvGaussianDoniachSinglett(prefix = "example_", independent_vars = ["x"])

In [5]:
peak.make_params()

name,value,initial value,min,max,vary,expression
example_amplitude,100.000000,None,0.00000000,inf,True,
example_sigma,0.20000000,None,0.00000000,inf,True,
example_gamma,0.02000000,None,-inf,inf,True,
example_gaussian_sigma,0.20000000,None,0.00000000,inf,True,
example_center,100.000000,None,0.00000000,inf,True,
example_gaussian_fwhm,0.47096000,None,-inf,inf,False,2*example_gaussian_sigma*1.1774
example_lorentzian_fwhm,0.41005962,None,-inf,inf,False,example_sigma*(2+example_gamma*2.5135+(example_gamma*3.6398)**4)


In [6]:
peak.set_param_hint("amplitude", value = 50, min = 0)

In [7]:
def get_satellite_row(name, expr):
    result = {
        "Parameter": name,
        "value": "",
        "min": "",
        "max": "",
        "vary": "False",
        "expr": expr,
    }
    return result

def get_rows(parent_prefix, satellite):
    full_prefix = parent_prefix + satellite['name'] + '_'
    rows = []
    rows.append(get_satellite_row(f"{full_prefix}amplitude",
                f"{parent_prefix}amplitude*{satellite['intensity']}/100"))
    rows.append(get_satellite_row(f"{full_prefix}sigma", f"{parent_prefix}sigma"))
    rows.append(get_satellite_row(f"{full_prefix}gamma", f"{parent_prefix}gamma"))
    rows.append(get_satellite_row(f"{full_prefix}gaussian_sigma", f"{parent_prefix}gaussian_sigma"))
    rows.append(get_satellite_row(f"{full_prefix}center",
                f"{parent_prefix}center+{satellite['position']}"))
        
    rows.append(get_satellite_row(f"{full_prefix}gaussian_fwhm",
               f'2*{full_prefix}gaussian_sigma*1.1774'))
    rows.append(get_satellite_row(f"{full_prefix}lorentzian_fwhm",
               f'{full_prefix}sigma*(2+{full_prefix}gamma*2.5135+({full_prefix}gamma*3.6398)**4)'))

    return rows

In [8]:
example_rows = get_rows("example_", satellites[1])
pd.DataFrame(example_rows)

,Parameter,value,min,max,vary,expr
0,example_alpha_3_amplitude,,,,False,example_amplitude*8.0/100
1,example_alpha_3_sigma,,,,False,example_sigma
2,example_alpha_3_gamma,,,,False,example_gamma
3,example_alpha_3_gaussian_sigma,,,,False,example_gaussian_sigma
4,example_alpha_3_center,,,,False,example_center+8.4
5,example_alpha_3_gaussian_fwhm,,,,False,2*example_alpha_3_gaussian_sigma*1.1774
6,example_alpha_3_lorentzian_fwhm,,,,False,example_alpha_3_sigma*(2+example_alpha_3_gamma...
